# 🖥️ Sigma Coordinate Vertical Grids on Unstructured Grids

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from matplotlib.animation import FuncAnimation

import parcels
from parcels._datasets.unstructured.generated import sigma_coordinate_lattice_dataset

# for interactive display of animations
plt.rcParams["animation.html"] = "jshtml"

## Build the channel mesh

We begin by constructing a 2D mesh that we will later extend into 3 dimensions.

In [ ]:
nx = 21  # nodes along and across the channel
n_layers = 10
x_nodes, _ = np.meshgrid(np.linspace(0.0, 40e3, nx), np.linspace(0.0, 10e3, nx), indexing="ij")
bottom_depth = 10.0 + 90.0 * np.log10(1.0 + 9.0 * x_nodes / 40e3)  # (x, y) array, in meters

ds_static = sigma_coordinate_lattice_dataset(nx, (0.0, 40e3), (0.0, 10e3), n_layers + 1, bottom_depth)
node_x, node_y = ds_static.uxgrid.node_lon.values, ds_static.uxgrid.node_lat.values
face_nodes = ds_static.uxgrid.face_node_connectivity.values

print(f"n_node={ds_static.uxgrid.n_node}, n_face={ds_static.uxgrid.n_face}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.triplot(node_x / 1e3, node_y / 1e3, face_nodes, color="k", linewidth=0.5)
ax.set_aspect("equal")
ax.set_xlabel("x [km]")
ax.set_ylabel("y [km]")
ax.set_title("Channel mesh")
plt.show()

## Build the vertical grid

Next we give the channel a terrain-following (sigma) vertical coordinate. The bottom depth $H$ increases logarithmically along the channel, from 10 m at the shallow left end to 100 m at the deep right end, so the bottom slopes steeply near the shallow end and flattens towards the deep end, typical of a coastal shelf. Following Parcels' convention that depth is positive downward, we define $\sigma = \frac{z + \eta}{H + \eta}$, so $\sigma = 0$ at the surface and $\sigma = 1$ at the bottom. Initially, we assume $\eta$ to be zero. We place 11 layer interfaces (10 layers) equally spaced in $\sigma$, so every node has its own column of interface depths $z = \sigma H$.

In [ ]:
sigma = np.linspace(0.0, 1.0, n_layers + 1)
z = ds_static["zf"].values[0]  # interface depths at every node, shape (zf, n_node)
H = z[-1]

In [ ]:
centre_line = np.isclose(node_y, 5e3) 
xs = node_x[centre_line] / 1e3

fig, (ax_sigma, ax_z) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
ax_sigma.plot(xs, np.broadcast_to(sigma, (xs.size, sigma.size)), "k", linewidth=0.8)
ax_sigma.set_ylabel(r"$\sigma$")
ax_sigma.set_title(r"Layer interfaces in $\sigma$ space")

ax_z.plot(xs, z[:, centre_line].T, "k", linewidth=0.8)
ax_z.set_ylabel("depth [m]")
ax_z.set_title("Layer interfaces in z space")

for ax in (ax_sigma, ax_z):
    ax.invert_yaxis()
    ax.set_xlabel("x [km]")
plt.tight_layout()
plt.show()

## Temporal Variation in $\eta$

Next, we add a fluctuating sea surface height field to the vertical grid. For illustrative purposes, we use a large amplitude sea surface height wave that's independent of the velocity field. In sigma space, the vertical grid remains exactly the same, since it accounts for $\eta$ when calculating the fractional vertical coordinate. In z space however, the change in $\eta$ causes fluctuations in the grid itself.

In [ ]:
t = np.arange(13) * 3600.0

# for illustrative purposes we put a 2m surface wave that propogates with time
wavelength, wave_period = 20e3, 6 * 3600.0
wave = 2 * np.sin(2.0 * np.pi * (x_nodes[np.newaxis] / wavelength - t[:, np.newaxis, np.newaxis] / wave_period))

ds = sigma_coordinate_lattice_dataset(nx, (0.0, 40e3), (0.0, 10e3), n_layers + 1, bottom_depth, eta=wave)
time = ds["time"].values
z_moving = ds["zf"].values  # (time, zf, n_node)
eta = -z_moving[:, 0, :]  # the surface interface sits at depth -eta

In [ ]:
fig, (ax_sigma, ax_z) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
ax_sigma.plot(xs, np.broadcast_to(sigma, (xs.size, sigma.size)), "k", linewidth=0.8)
ax_sigma.set_ylabel(r"$\sigma$")
ax_sigma.set_title(r"Layer interfaces in $\sigma$ space")

z_lines = ax_z.plot(xs, z_moving[0][:, centre_line].T, "k", linewidth=0.8)
ax_z.set_ylabel("depth [m]")
z_title = ax_z.set_title(f"Layer interfaces in z space at {time[0]}")

for ax in (ax_sigma, ax_z):
    ax.invert_yaxis()
    ax.set_xlabel("x [km]")
plt.tight_layout()

def animate(snapshot):
    for line, interface_depths in zip(z_lines, z_moving[snapshot][:, centre_line]):
        line.set_ydata(interface_depths)
    z_title.set_text(f"Layer interfaces in z space at {time[snapshot]}")

anim = FuncAnimation(fig, animate, frames=time.size, interval=500)
plt.close()
anim

## Building the dataset and `FieldSet`

We prescribe a uniform along channel velocity of $u_0 = 0.5$ m/s, with $v = w = 0$. All three components sit on the mesh nodes at the layer interfaces (`zf`), so Parcels interpolates them barycentrically in the horizontal and linearly in the vertical.

To advect the particles on sigma grid, for which the z grid varies in time, we use a 3D `zf` coordinate with dimensions of `(time, zf, n_node)`. Parcels then uses it to find each particle's own column at the current time, before searching that column for the particle's layer. `parcels.FieldSet.from_ugrid_conventions()` also expects a `zc` dimension for the layer centres, so we add the layer-centre depths as well. We pass `mesh="flat"` because the coordinates are Cartesian meters.

In [ ]:
u0 = 0.5
ds["U"].values[:] = u0
ds

In [ ]:
fieldset = parcels.FieldSet.from_ugrid_conventions(ds, mesh="flat")
fieldset.describe()

## Release the particles

We release 10 particles at the first snapshot near the shallow end of the channel, at $x = 2$ km on the centre line, at depths from 3 m to 22 m. The water column there is about 22.5 m deep, and the shallowest particle starts below the deepest wave trough (2 m), so every particle will remain in the water column throughout the simulation.

In [ ]:
n_particles = 10
x0 = np.full(n_particles, 2e3)
y0 = np.full(n_particles, 5e3)
z0 = np.linspace(3.0, 22.0, n_particles)  # depths in m

pset = parcels.ParticleSet(
    fieldset=fieldset, pclass=parcels.Particle, t=np.full(n_particles, time[0]), z=z0, y=y0, x=x0
)

In [ ]:
x_line = node_x[centre_line]
H0 = np.interp(x0, x_line, H[centre_line])
eta0 = np.interp(x0, x_line, eta[0, centre_line])
sigma0 = (z0 + eta0) / (H0 + eta0)  # initial sigma coordinate of each particle

colors = [f"C{p}" for p in range(n_particles)]

fig, (ax_sigma, ax_z) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
ax_sigma.plot(xs, np.broadcast_to(sigma, (xs.size, sigma.size)), "k", linewidth=0.8)
ax_sigma.scatter(x0 / 1e3, sigma0, color=colors, zorder=3)
ax_sigma.set_ylabel(r"$\sigma$")
ax_sigma.set_title(r"Initial particle positions in $\sigma$ space")

ax_z.plot(xs, z_moving[0][:, centre_line].T, "k", linewidth=0.8)
ax_z.scatter(x0 / 1e3, z0, color=colors, zorder=3)
ax_z.set_ylabel("depth [m]")
ax_z.set_title(f"Initial particle positions in z space at {time[0]}")

for ax in (ax_sigma, ax_z):
    ax.invert_yaxis()
    ax.set_xlabel("x [km]")
plt.tight_layout()
plt.show()

## Advect the particles

We advect the particles for 12 hours using the 3D fourth-order Runge-Kutta kernel `parcels.kernels.AdvectionRK4_3D` with a 5-minute time step, and write their positions every 30 minutes. 

In [ ]:
outputfile = parcels.ParticleFile(path="sigma_particles.parquet", outputdt=np.timedelta64(30, "m"), mode="w")

pset.execute(
    [parcels.kernels.AdvectionRK4_3D],
    runtime=np.timedelta64(12, "h"),
    dt=np.timedelta64(5, "m"),
    output_file=outputfile,
)

In [ ]:
df = parcels.read_particlefile("sigma_particles.parquet").sort(["t", "particle_id"])
output_times = df["t"].unique().sort()
n_frames = len(output_times)
px = df["x"].to_numpy().reshape(n_frames, n_particles)
pz = df["z"].to_numpy().reshape(n_frames, n_particles)

# sigma coordinate of every particle at every output time, on the latest grid snapshot at or before that time
snapshots = np.searchsorted(time, output_times.to_numpy(), side="right") - 1
psigma = np.empty_like(pz)
for frame, snapshot in enumerate(snapshots):
    eta_p = np.interp(px[frame], x_line, eta[snapshot, centre_line])
    psigma[frame] = (pz[frame] + eta_p) / (np.interp(px[frame], x_line, H[centre_line]) + eta_p)

colors = [f"C{p}" for p in range(n_particles)]

fig, (ax_sigma, ax_z) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
ax_sigma.plot(xs, np.broadcast_to(sigma, (xs.size, sigma.size)), "k", linewidth=0.8)
z_lines = ax_z.plot(xs, z_moving[0][:, centre_line].T, "k", linewidth=0.8)
sigma_trails = [ax_sigma.plot([], [], color=c, linewidth=1.5)[0] for c in colors]
z_trails = [ax_z.plot([], [], color=c, linewidth=1.5)[0] for c in colors]
sigma_dots = ax_sigma.scatter(px[0] / 1e3, psigma[0], c=colors, zorder=3)
z_dots = ax_z.scatter(px[0] / 1e3, pz[0], c=colors, zorder=3)

ax_sigma.set_ylabel(r"$\sigma$")
ax_sigma.set_title(r"Particles in $\sigma$ space")
ax_z.set_ylabel("depth [m]")
z_title = ax_z.set_title("")
for ax in (ax_sigma, ax_z):
    ax.invert_yaxis()
    ax.set_xlabel("x [km]")
plt.tight_layout()

def animate(frame):
    for p in range(n_particles):
        sigma_trails[p].set_data(px[: frame + 1, p] / 1e3, psigma[: frame + 1, p])
        z_trails[p].set_data(px[: frame + 1, p] / 1e3, pz[: frame + 1, p])
    sigma_dots.set_offsets(np.c_[px[frame] / 1e3, psigma[frame]])
    z_dots.set_offsets(np.c_[px[frame] / 1e3, pz[frame]])
    for line, interface_depths in zip(z_lines, z_moving[snapshots[frame]][:, centre_line]):
        line.set_ydata(interface_depths)
    z_title.set_text(f"Particles in z space at {output_times[frame]}")


animate(0)
anim = FuncAnimation(fig, animate, frames=n_frames, interval=300)
plt.close()
anim

We can see that, as expected, the particles move horizontally in x and cross the terrain following depth levels. However, in z space, they maintain their exact height as expected.